## To-Do:

1. Make tool methods: update, save document:
   - update: update the document in the global variable
   - save: save the document in a txt file
2. make an array with the tool call functions
3. Make the main LLM agent function
4. make should_continue function to determine the conditional logic with the LLM process
5. make a helper function that prints tool result message in a tidy way 
6. Make a graph
7. Make the executive function

In [1]:
from typing import TypedDict, Annotated, Sequence
from dotenv import load_dotenv
import os

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

In [2]:
load_dotenv()

True

In [3]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [4]:
document_content = ""

In [5]:
@tool 
def update(doc: str) -> str:
    """
    Update document to the global document_content variable.
    """
    global document_content
    
    if not doc:
        print("ARG error: doc wasn't provided")
        return "ARG error: doc wasn't provided"
    
    document_content = doc
    
    return f"Document was updated successfully!: {document_content}"


@tool
def save(filename: str) -> str:
    """
    Save the updated document_content into a txt file with the most appropriate file name.
    """
    global document_content

    if not filename.endswith(".txt"):
        filename = f"{filename}.txt"

    try:
        with open(filename, "w") as f:
            f.write(document_content)

        return f"Document was saved into the file: {filename}" 
    except Exception as e:
        return f"Error saving document: {e}"

In [6]:
tools = [update, save]

llm = ChatOpenAI(model="gpt-5-mini").bind_tools(tools)

In [7]:
def llm_agent(state: AgentState) -> AgentState:
    """
    Node that processes all the LLM reasoning processes and handles the conversation history.
    """

    system_message = SystemMessage(content=f"""
    You are a helpful writing assistant and drafter that helps modify, update, and finally save a document. 

    - If you need to update a document, use the update tool. 
    - If you need to save the document, use the save tool. The user will ask "save it" or any similar instruction and it's your job to save the document into a txt file.
    - Do not speak any additional comments. Just return the document content in your response.
    - Make sure to always show the current document state after modifications. 

    The current document content is: {document_content}
    """)

    if not state["messages"]:
        user_input = input(f"What document would you like to create? ")
        human_message = HumanMessage(content=user_input)
    else:
        user_input = input(f"How do you want to modify the document? ")
        human_message = HumanMessage(content=user_input)

    all_messages = [system_message] + list(state["messages"]) + [human_message]

    response = llm.invoke(all_messages)
    # print("AI RES: ", response.content)

    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"TOOL USE: {[tc["name"] for tc in response.tool_calls]}")

    return {"messages": list(state["messages"]) + [human_message, response]}

In [8]:
def should_continue(state: AgentState) -> str:
    """
    Determine if it should continue or end the conversation.
    """

    messages = state["messages"]
    
    if not messages:
        return "continue"

    for message in reversed(messages):
        if (isinstance(message, ToolMessage) and
            "document" in message.content.lower() and
            "saved" in message.content.lower()
           ): 
            return "end"

    return "continue"

In [9]:
graph = StateGraph(AgentState)

graph.add_node("llm_agent", llm_agent)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "llm_agent")
graph.add_edge("llm_agent", "tools")

graph.add_conditional_edges(
    source="tools", 
    path=should_continue,
    path_map={
        "continue": "llm_agent",
        "end": END
    }
)

app = graph.compile()

In [10]:
app.stream

<bound method Pregel.stream of <langgraph.graph.state.CompiledStateGraph object at 0x10d24b620>>

In [11]:
def print_messages(messages):
    """
    Print ToolMessage in a prettier way.
    """
    if not messages:
        return 

    for message in messages[-3:]:
        if isinstance(message, ToolMessage):
            print(f"TOOL RESPONSE: {message.content}")

In [12]:
def run_conversation_agent():
    print("\n ===== DRAFTER =====")
    
    state = {"messages": []}

    for step in app.stream(state, stream_mode="values"):
        if "messages" in step:
            print_messages(step["messages"])

    print("\n ===== DRAFTER FINISHED =====")

In [ ]:
if __name__ == "__main__":
    run_conversation_agent()


 ===== DRAFTER =====


What document would you like to create?  write an email for my subscription cancelling at Netflix


TOOL USE: ['update']
TOOL RESPONSE: Document was updated successfully!: Subject: Request to Cancel Netflix Subscription (Account: [your email])

Dear Netflix Customer Support,

I am writing to request cancellation of my Netflix subscription associated with the email address [your email] and account name [your name]. Please cancel my subscription effective immediately and stop any future billing.

Subscription details:
- Account email: [your email]
- Full name: [your full name]
- Plan: [Basic / Standard / Premium]
- Last billing date: [last billing date]

Please confirm when the cancellation has been processed and provide a cancellation confirmation number or email. Also confirm that I will not be billed going forward and indicate the final billing date and any applicable refunds or prorated charges.

Thank you,

[Your full name]
[Phone number (optional)]

